# 🤖 Notebook 03 – Model Training & Evaluation
**Project:** Customer Churn Prediction

---
This notebook covers:
- Training Logistic Regression, Random Forest, and XGBoost
- MLflow experiment tracking
- Model comparison (Accuracy, Precision, Recall, F1, AUC-ROC)
- Confusion matrices and ROC curves
- Feature importance analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

## 1. Load & Preprocess Data

In [ ]:
from src.data.load_data import load_raw_data, load_config
from src.data.preprocess import run_preprocessing_pipeline

config = load_config()
df = load_raw_data()
X_train, X_test, y_train, y_test, scaler = run_preprocessing_pipeline(df)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Train All Models with MLflow Tracking

In [ ]:
from src.models.train import train_and_track

best_model_name = train_and_track(X_train, X_test, y_train, y_test, config)
print(f'\n🏆 Best model: {best_model_name}')

## 3. Model Evaluation & Comparison

In [ ]:
import joblib
from src.models.train import get_models
from src.models.evaluate import evaluate_model, compare_models

models = get_models(config)
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)  # re-train for evaluation
    metrics = evaluate_model(model, X_test, y_test, model_name=name)
    results[name] = metrics

comparison_df = compare_models(results)
comparison_df

## 4. Confusion Matrices

In [ ]:
from src.visualization.plots import plot_confusion_matrix

for name, model in models.items():
    y_pred = model.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, model_name=name)
    print(f'✅ Confusion matrix saved for {name}')

## 5. ROC Curves

In [ ]:
from src.visualization.plots import plot_roc_curves

roc_data = {}
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    roc_data[name] = (y_test, y_prob)

plot_roc_curves(roc_data)
print('✅ ROC curves saved')

## 6. Feature Importance

In [ ]:
from src.visualization.plots import plot_feature_importance

feature_names = X_train.columns.tolist()
plot_feature_importance(models['RandomForest'], feature_names, 'RandomForest')
plot_feature_importance(models['XGBoost'], feature_names, 'XGBoost')
print('✅ Feature importance plots saved')

## 7. Summary

Run `mlflow ui` in the project root to explore all experiment runs interactively.

The best model is saved to `models/best_model.pkl` and will be automatically loaded by the FastAPI server.

➡️ Start the API: `uvicorn api.main:app --reload`